<a href="https://colab.research.google.com/github/Ravenveil/INTRA-repro/blob/Colab/INTRA_repro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INTRA 论文复现笔记本

## INTRA 论文复现 - 流程总览

本笔记本旨在复现 INTRA 论文的相关实验。整个流程大致分为以下几个步骤：

*   **第0步：运行环境检查**：验证 Python、PyTorch、CUDA、磁盘空间和必要的依赖库。
*   **第1步：环境准备**：克隆 INTRA 代码库，拉取 `git-lfs` 管理的差分权重文件，并安装项目所需的 Python 包。
*   **第2步：构建 INTRA 模型 Checkpoint**：将下载的差分权重应用到 Hugging Face 的基础模型（`google/gemma-2b`）上，生成可用的 INTRA 模型权重。
*   **第3步：玩具演示**：在小数据集上运行模型，验证完整流水线的正确性，避免在大型实验上浪费资源。（原 notebook 中的第4步）
*   **第4步：完整基准测试**：重建 CLaRa 1亿文档池，编码向量检索库，并在多个基准测试上评估模型性能。（原 notebook 中的第5-7步）

**重要提示**：Colab 运行时环境是临时的。当会话断开或重启时，所有文件都会丢失。如果您想保存实验结果或修改后的代码，请务必将其下载到本地或保存到 Google Drive。

### 第0步：运行环境检查

运行此单元格以检查当前的 Python 环境、PyTorch/CUDA 设置、磁盘空间以及主要的 Python 依赖包。这将帮助您判断当前环境是否满足 INTRA 复现的硬件和软件要求。

In [35]:
# ================================================================
# INTRA 论文复现 - 第0步：运行环境检查
# 论文：Retrieval from Within (注意力机制的内在检索能力)
# 代码库：https://github.com/paper-submissions/INTRA
# ================================================================
# 复现等级说明：
#   L0: 只读代码，任何机器均可
#   L1: 玩具演示，需要 16GB 显存（仅 21 篇文档）
#   L2: 完整基准测试，需要 80GB 显存（CLaRa 1亿文档池）
# ================================================================

import subprocess, sys, os, shutil


def check_env():
    print("=" * 62)
    print("INTRA 复现环境检查")
    print("=" * 62)

    # ---------- 1. 检查 Python 版本 ----------
    print(f"\n[Python] {sys.version}")

    # ---------- 2. 检查 PyTorch + CUDA ----------
    try:
        import torch
        print(f"[PyTorch] {torch.__version__}")
        cuda_ok = torch.cuda.is_available()
        print(f"[CUDA 可用] {cuda_ok}")

        if cuda_ok:
            n = torch.cuda.device_count()
            print(f"[GPU 数量] {n}")
            for i in range(n):
                p = torch.cuda.get_device_properties(i)
                vram = p.total_memory / 1024**3          # 转换为 GB
                cc = f"{p.major}.{p.minor}"              # 计算能力版本号
                bf16 = p.major >= 8                      # Ampere 架构(8.0+)才支持 bfloat16
                # 根据显存大小判断可复现的等级
                tier = ("L2 Full (>=80GB)" if vram >= 80
                        else ("L1 Toy Demo (>=16GB)" if vram >= 16
                              else "L0 only (<16GB)"))
                print(f"  GPU[{i}]: {p.name}")
                print(f"  显存: {vram:.1f} GB -> {tier}")
                print(f"  架构: CC {cc}  bf16={'支持' if bf16 else '需要 Ampere 8.0+'}")

        # FlexAttention 要求 PyTorch >= 2.8
        parts = torch.__version__.split('.')
        maj, minor_v = int(parts[0]), int(parts[1].split('+')[0])
        ok = maj > 2 or (maj == 2 and minor_v >= 8)
        print(f"[PyTorch >= 2.8] {'OK' if ok else '版本不足，FlexAttention 需要 >= 2.8'}")

    except ImportError as e:
        print(f"[PyTorch] 未安装: {e}")

    # ---------- 3. 检查磁盘空间 ----------
    disk_path = '/home/aistudio'
    if not os.path.exists(disk_path):
        disk_path = os.getcwd()
    try:
        total, used, free = shutil.disk_usage(disk_path)
        free_gb = free / 1024**3
        total_gb = total / 1024**3
        print(f"\n[磁盘 {disk_path}] 剩余: {free_gb:.1f} GB / 总计: {total_gb:.1f} GB")
        print(f"  状态: {'OK (>=100GB)' if free_gb>=100 else '警告 <100GB' if free_gb>=30 else '严重不足 <30GB'}")
    except Exception as e:
        print(f"\n[磁盘检查] 无法检测 {disk_path}，原因：{e}")

    # ---------- 4. 检查关键 Python 依赖包 ----------
    print("\n[依赖包检查]")
    deps = {
        'transformers'    : '4.40',
        'datasets'        : '2.0',
        'numpy'           : '1.24',
        'tqdm'            : '4.0',
        'huggingface_hub' : '0.21',
    }
    for dep, min_ver in deps.items():
        try:
            mod = __import__(dep)
            ver = getattr(mod, '__version__', '?')
            print(f"  {dep:20s}: OK  {ver}")
        except ImportError:
            print(f"  {dep:20s}: 缺失！")

    # ---------- 5. 检查 git-lfs（用于拉取差分权重文件）----------
    try:
        r = subprocess.run(['git', 'lfs', 'version'],
                           capture_output=True, text=True, timeout=5)
        if r.returncode == 0:
            print(f"\n[git-lfs] OK  {r.stdout.strip()}")
        else:
            print("\n[git-lfs] 未安装 — 克隆 diff 前必须安装！")
    except Exception as e:
        print(f"\n[git-lfs] 错误: {e}")

    print("\n" + "=" * 62)
    print("环境检查完毕，可根据显存选择 L1 玩具演示或 L2 完整基准。")
    print("=" * 62)

check_env()

INTRA 复现环境检查

[Python] 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[PyTorch] 2.11.0+cpu
[CUDA 可用] False
[PyTorch >= 2.8] OK

[磁盘 /home/aistudio] 剩余: 65.5 GB / 总计: 107.7 GB
  状态: 警告 <100GB

[依赖包检查]
  transformers        : OK  5.12.0
  datasets            : OK  4.0.0
  numpy               : OK  2.0.2
  tqdm                : OK  4.67.3
  huggingface_hub     : OK  1.19.0

[git-lfs] OK  git-lfs/3.7.1 (GitHub; linux amd64; go 1.26.0)

环境检查完毕，可根据显存选择 L1 玩具演示或 L2 完整基准。


In [33]:
check_env()

INTRA 复现环境检查

[Python] 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[PyTorch] 2.11.0+cpu
[CUDA 可用] False
[PyTorch >= 2.8] OK

[磁盘 /home/aistudio] 剩余: 65.5 GB / 总计: 107.7 GB
  状态: 警告 <100GB

[依赖包检查]
  transformers        : OK  5.12.0
  datasets            : OK  4.0.0
  numpy               : OK  2.0.2
  tqdm                : OK  4.67.3
  huggingface_hub     : OK  1.19.0

[git-lfs] OK  git-lfs/3.7.1 (GitHub; linux amd64; go 1.26.0)

环境检查完毕，可根据显存选择 L1 玩具演示或 L2 完整基准。


### 第1步：环境准备 - 克隆仓库、拉取 LFS 和安装依赖

此步骤将克隆 INTRA 仓库，下载所有必要的 Git LFS 文件（包括差分权重），并安装 Python 依赖项。

In [36]:
# ================================================================
# INTRA 复现 - 第1步：克隆代码库并通过 git-lfs 拉取差分权重
# ================================================================
# 注意：只需运行一次。
# assets/ 目录下的差分权重通过 git-lfs 存储。
# 若未执行 'git lfs pull'，你只会得到 ~500 字节的指针文件，
# 导致 build_checkpoint.py 静默失败！
# ================================================================

import os, subprocess

WORK_DIR  = "/home/aistudio"
INTRA_DIR = os.path.join(WORK_DIR, "INTRA")   # 仓库本地路径


def run(cmd, cwd=None, check=True):
    """执行 shell 命令并实时打印输出，减少长时间挂起时的误触发中断。"""
    print(f">>> {cmd}")
    proc = subprocess.Popen(
        cmd, shell=True, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    stdout_lines = []
    try:
        for line in proc.stdout:
            print(line, end="")
            stdout_lines.append(line)
        proc.wait()
    except KeyboardInterrupt:
        proc.kill()
        raise
    result = subprocess.CompletedProcess(proc.args, proc.returncode, stdout="".join(stdout_lines))
    if check and result.returncode != 0:
        raise RuntimeError(f"命令失败 (rc={result.returncode}): {cmd}")
    return result

# 1-A. 克隆仓库（若已存在则跳过）
if not os.path.isdir(INTRA_DIR):
    print("[Step 1] 开始克隆仓库，请耐心等待，不要中断。")
    run(f"git clone --depth 1 https://github.com/paper-submissions/INTRA {INTRA_DIR}")
else:
    print(f"[Step 1] 已存在仓库，跳过克隆: {INTRA_DIR}")


[Step 1] 已存在仓库，跳过克隆: /home/aistudio/INTRA


#### 1-A. 克隆仓库与验证文件夹存在

此步骤将克隆 INTRA 仓库，并在完成后检查 `INTRA` 文件夹是否已成功存在于 `/home/aistudio/INTRA` 路径。请运行 `ppZjDTv1qdxy` 和 `0d52e85d` 单元格。

In [37]:
import os

INTRA_DIR = "/home/aistudio/INTRA"

if os.path.exists(INTRA_DIR):
    print(f"INTRA 文件夹已存在于: {INTRA_DIR}")
    print("以下是该文件夹的部分内容：")
    !ls -F {INTRA_DIR}
else:
    print(f"错误：INTRA 文件夹未找到于: {INTRA_DIR}")

INTRA 文件夹已存在于: /home/aistudio/INTRA
以下是该文件夹的部分内容：
assets/       data/	  intra/	   LICENSE	   README.md  tests/
checkpoints/  hf_models/  intra.egg-info/  pyproject.toml  scripts/


#### 1-B. 拉取 Git LFS 文件

此步骤将安装 `git-lfs` 并拉取 INTRA 仓库中通过 Git LFS 管理的实际差分权重文件。这些文件是构建模型 Checkpoint 所必需的。请运行 `wvQjb178q3BO` 单元格。

In [38]:
# 1-B. 拉取真实差分权重（关键！）
if os.path.isdir(INTRA_DIR):
    print("[Step 2] 安装 git-lfs 并拉取 LFS 对象")
    run("git lfs install", cwd=INTRA_DIR)
    run("git lfs pull", cwd=INTRA_DIR)
else:
    raise RuntimeError(f"仓库目录不存在：{INTRA_DIR}")


[Step 2] 安装 git-lfs 并拉取 LFS 对象
>>> git lfs install
Updated Git hooks.
Git LFS initialized.
>>> git lfs pull


#### 1-C. 验证差分权重是否为真实文件（而非指针）及安装 Python 包

此单元格将检查 Git LFS 文件是否已成功下载为真实数据（而不是指针），并安装项目所需的 Python 依赖包。

In [39]:
# 1-C. 验证差分权重是否为真实文件（而非指针）
diff_index = os.path.join(
    INTRA_DIR, "assets",
    "intra_diff_t5gemma2_4b-4b_Lp7", "intra_diff.index.json"
)
if os.path.exists(diff_index):
    size = os.path.getsize(diff_index)
    print(f"[Step 3] 差分索引存在，大小={size} 字节")
    if size < 1000:
        print("警告：差分索引疑似仍是 git-lfs 指针，请检查 'git lfs pull'")
    else:
        print("OK：差分索引为真实文件")
else:
    print(f"错误：差分索引不存在于 {diff_index}")

# 1-D. 以可编辑模式安装 Python 包（含数据处理扩展）
print("[Step 4] 安装 Python 依赖包（包含 data 扩展）")
run(f"pip install -e '.[data]' -q", cwd=INTRA_DIR)
print("\n第1步完成：仓库已克隆，差分权重已拉取，包已安装。")

[Step 3] 差分索引存在，大小=41486 字节
OK：差分索引为真实文件
[Step 4] 安装 Python 依赖包（包含 data 扩展）
>>> pip install -e '.[data]' -q

第1步完成：仓库已克隆，差分权重已拉取，包已安装。


### 第2步：构建 INTRA 模型 Checkpoint (最关键一步)

这里涉及几个关键概念：

1.  **差分文件 (`diff files`)**：这些不是完整的模型权重，而是记录了如何将一个基础模型（例如 `google/gemma-2b`）修改成 INTRA 模型的“差异”部分。它们通过 `git-lfs` 管理，通常是较大的二进制文件。
2.  **Checkpoint**：这是包含完整模型权重的文件或目录。在 INTRA 项目中，最终可用的 INTRA 模型权重将以 Checkpoint 的形式保存。
3.  **合并过程 (`build_checkpoint.py`)**：INTRA 项目提供了 `scripts/build_checkpoint.py` 脚本来完成“合并”或“构建”Checkpoint 的任务。它会首先下载或加载指定的 **基础模型** (例如 `google/gemma-2b`)，然后将这些 **差分文件** 应用到基础模型上，从而生成一个完整的、可直接用于推理的 INTRA 模型 **Checkpoint**。这个过程确保了复现的一致性和效率。

**重要提示**：在执行 `build_checkpoint.py` 之前，我们需要确保基础模型 `google/gemma-2b` 已经下载到本地。如果未下载，脚本会因为找不到 `safetensors` 文件而报错。下面的单元格将先下载 `google/gemma-2b`，然后执行构建 Checkpoint 的操作。

#### 2-A. 下载基础模型 `google/gemma-2b`

`build_checkpoint.py` 脚本需要本地的基础模型文件。此步骤将使用 `transformers` 库将 `google/gemma-2b` 模型下载到 `INTRA/hf_models/gemma-2b` 目录下。

**重要提示：下载 `google/gemma-2b` 模型需要完成以下两个步骤：**

1.  **接受模型的使用条款**：您需要访问模型页面 `https://huggingface.co/google/gemma-2b`，登录您的 Hugging Face 账号，并阅读、接受其使用条款。通常，这会通过点击页面上的一个按钮来完成，完成后您会获得该模型的访问权限。
2.  **提供 Hugging Face 认证令牌**：下载模型时，`transformers` 库需要通过您的 Hugging Face 个人访问令牌 (Personal Access Token, PAT) 进行身份验证。请按以下步骤操作：
    *   前往 `https://huggingface.co/settings/tokens` 创建一个具有 Read 权限的 PAT。
    *   在 Colab 左侧面板中找到 `🔒` (Secrets) 图标，点击它并添加一个名为 `HF_TOKEN` 的新 Secret，将您在 Hugging Face 生成的 PAT 粘贴进去。
    *   代码中的 `token=True` 会自动尝试使用这个 `HF_TOKEN` Secret 进行认证。如果仍有问题，可以尝试将 `token=True` 替换为 `token=os.environ.get('HF_TOKEN')`。

完成上述步骤后，请运行下面的 Python 单元格来下载模型。

In [42]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import userdata # 导入 userdata

INTRA_DIR = "/home/aistudio/INTRA"
BASE_MODEL_LOCAL_PATH = os.path.join(INTRA_DIR, "hf_models", "gemma-2b")

if not os.path.exists(BASE_MODEL_LOCAL_PATH):
    print(f"正在下载基础模型 google/gemma-2b 到 {BASE_MODEL_LOCAL_PATH}...")

    # 明确从 Colab Secrets 获取 HF_TOKEN
    hf_token = userdata.get('HF_TOKEN') # 从 Colab Secrets 中获取 HF_TOKEN

    if not hf_token:
        raise ValueError("HF_TOKEN 未在 Colab Secrets 中设置。请按照指引添加。")

    # 使用获取到的 token 进行认证
    tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b", token=hf_token)
    model = AutoModelForCausalLM.from_pretrained("google/gemma-2b", token=hf_token)

    tokenizer.save_pretrained(BASE_MODEL_LOCAL_PATH)
    model.save_pretrained(BASE_MODEL_LOCAL_PATH)
    print("基础模型下载并保存完成。")
else:
    print(f"基础模型已存在于 {BASE_MODEL_LOCAL_PATH}，跳过下载。")


正在下载基础模型 google/gemma-2b 到 /home/aistudio/INTRA/hf_models/gemma-2b...


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

基础模型下载并保存完成。


### 验证 `google/gemma-2b` 本地保存的文件内容

在 `dca9f772` 单元格显示下载成功后，但 `build_checkpoint.py` 仍然报错 `FileNotFoundError: No safetensors found` 的情况下，运行此单元格以检查 `BASE_MODEL_LOCAL_PATH` (`/home/aistudio/INTRA/hf_models/gemma-2b`) 目录下实际保存的文件。我们需要确认 `safetensors` 文件是否存在，以及它们的文件名和大小。

In [43]:
import os

INTRA_DIR = "/home/aistudio/INTRA"
BASE_MODEL_LOCAL_PATH = os.path.join(INTRA_DIR, "hf_models", "gemma-2b")

print(f"正在检查目录: {BASE_MODEL_LOCAL_PATH}")

if os.path.exists(BASE_MODEL_LOCAL_PATH):
    print("目录存在。以下是其内容：")
    files_in_dir = os.listdir(BASE_MODEL_LOCAL_PATH)

    safetensors_found = False
    for f in files_in_dir:
        file_path = os.path.join(BASE_MODEL_LOCAL_PATH, f)
        if os.path.isfile(file_path):
            file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
            print(f"  - {f} (大小: {file_size_mb:.2f} MB)")
            if f.endswith('.safetensors'):
                safetensors_found = True
        else:
            print(f"  - {f}/ (这是一个子目录)")

    if safetensors_found:
        print("\n至少检测到一个 '.safetensors' 文件。这表明模型权重已保存。")
        print("如果 build_checkpoint.py 仍然报错，可能需要检查脚本如何加载这些文件。")
    else:
        print("\n**警告：未检测到任何 '.safetensors' 文件！**")
        print("这表明模型权重并未正确保存。请尝试重新运行下载单元格 `dca9f772`，并确保其完整执行。")

else:
    print(f"错误：目录 {BASE_MODEL_LOCAL_PATH} 不存在。请确保基础模型已下载。")

正在检查目录: /home/aistudio/INTRA/hf_models/gemma-2b
目录存在。以下是其内容：
  - config.json (大小: 0.00 MB)
  - tokenizer_config.json (大小: 0.00 MB)
  - model.safetensors (大小: 4780.16 MB)
  - tokenizer.json (大小: 32.76 MB)
  - generation_config.json (大小: 0.00 MB)

至少检测到一个 '.safetensors' 文件。这表明模型权重已保存。
如果 build_checkpoint.py 仍然报错，可能需要检查脚本如何加载这些文件。


### 可选：删除损坏的基础模型缓存以强制重新下载

由于 `config.json` 和 `generation_config.json` 文件大小为 0，这表明基础模型下载或保存不完整。运行此单元格将删除 `google/gemma-2b` 的本地缓存目录。删除后，请**务必重新运行 `dca9f772` 单元格**以确保所有模型文件（包括配置文件）都被完整下载和保存。

In [41]:
import os
import shutil

INTRA_DIR = "/home/aistudio/INTRA"
BASE_MODEL_LOCAL_PATH = os.path.join(INTRA_DIR, "hf_models", "gemma-2b")

if os.path.exists(BASE_MODEL_LOCAL_PATH):
    print(f"正在删除基础模型缓存目录: {BASE_MODEL_LOCAL_PATH}...")
    shutil.rmtree(BASE_MODEL_LOCAL_PATH)
    print("基础模型缓存目录已删除。请现在重新运行 `dca9f772` 单元格进行完整下载。")
else:
    print(f"基础模型缓存目录 {BASE_MODEL_LOCAL_PATH} 不存在，无需删除。")

正在删除基础模型缓存目录: /home/aistudio/INTRA/hf_models/gemma-2b...
基础模型缓存目录已删除。请现在重新运行 `dca9f772` 单元格进行完整下载。


#### 2-B. 构建 INTRA 模型 Checkpoint

此步骤将使用下载到本地的基础模型和 Git LFS 拉取的差分权重来构建最终的 INTRA 模型 Checkpoint。此过程涉及以下关键概念：

1.  **差分文件 (`diff files`)**：这些不是完整的模型权重，而是记录了如何将一个基础模型（例如 `google/gemma-2b`）修改成 INTRA 模型的“差异”部分。它们通过 `git-lfs` 管理，通常是较大的二进制文件。
2.  **Checkpoint**：这是包含完整模型权重的文件或目录。在 INTRA 项目中，最终可用的 INTRA 模型权重将以 Checkpoint 的形式保存。
3.  **合并过程 (`build_checkpoint.py`)**：INTRA 项目提供了 `scripts/build_checkpoint.py` 脚本来完成“合并”或“构建”Checkpoint 的任务。它会首先加载或下载指定的 **基础模型** (例如 `google/gemma-2b`)，然后将这些 **差分文件** 应用到基础模型上，从而生成一个完整的、可直接用于推理的 INTRA 模型 **Checkpoint**。这个过程确保了复现的一致性和效率。

**重要提示**：在执行此单元格之前，请确保 `dca9f772` 单元格（下载基础模型）已完整运行成功，并且基础模型已下载到 `BASE_MODEL_LOCAL_PATH`。如果此单元格显示“Checkpoint 已存在”，但您怀疑它不完整或有误，可以运行我提供的下一个单元格来删除现有 Checkpoint，强制重新构建。

In [44]:
import os

# 确保 run 函数可用 (它在 ppZjDTv1qdxy 单元格中定义)
# 如果您跳过了 ppZjDTv1qdxy 单元格，请先运行它。

INTRA_DIR = "/home/aistudio/INTRA"
CKPT_DIR = os.path.join(INTRA_DIR, "checkpoints", "t5gemma2_4b-4b_Lp7")
diff_dir_path = os.path.join(INTRA_DIR, "assets", "intra_diff_t5gemma2_4b-4b_Lp7")
BASE_MODEL_LOCAL_PATH = os.path.join(INTRA_DIR, "hf_models", "gemma-2b") # 确保与 2-A 步骤中的路径一致

if not os.path.exists(CKPT_DIR):
    print("[Step 2-B] 正在构建 INTRA 模型 checkpoint...")
    # 运行构建脚本，提供所有必需参数，使用本地基础模型路径
    build_cmd = (
        f"python scripts/build_checkpoint.py "
        f"--hf-assets {BASE_MODEL_LOCAL_PATH} "  # 使用本地下载的模型路径
        f"--diff-dir {diff_dir_path} "
        f"--output {CKPT_DIR}"
    )
    result = run(build_cmd, cwd=INTRA_DIR)
    print(result.stdout[-3000:]) # 打印最后3000字符的输出
    if result.returncode == 0:
        print("\nCheckpoint 构建成功！")
    else:
        print(f"\nCheckpoint 构建失败 (rc={result.returncode})，请查看上方输出。")
else:
    print(f"[Step 2-B] Checkpoint 已存在于: {CKPT_DIR}，跳过构建。")


[Step 2-B] Checkpoint 已存在于: /home/aistudio/INTRA/checkpoints/t5gemma2_4b-4b_Lp7，跳过构建。


### 可选：删除现有 Checkpoint 目录以强制重新构建

如果上一步显示“Checkpoint 已存在，跳过构建”，但您怀疑现有 Checkpoint 不完整或已损坏（例如，因为之前的基础模型下载没有完成），您可以选择运行此单元格来删除 `/home/aistudio/INTRA/checkpoints/t5gemma2_4b-4b_Lp7` 目录。删除后，再次运行 `09e59277` 单元格，就会触发完整的 Checkpoint 构建过程。

In [27]:
import os
import shutil

INTRA_DIR = "/home/aistudio/INTRA"
CKPT_DIR = os.path.join(INTRA_DIR, "checkpoints", "t5gemma2_4b-4b_Lp7")

if os.path.exists(CKPT_DIR):
    print(f"正在删除现有 Checkpoint 目录: {CKPT_DIR}...")
    shutil.rmtree(CKPT_DIR)
    print("Checkpoint 目录已删除。请现在再次运行构建 Checkpoint 的单元格 (09e59277)。")
else:
    print(f"Checkpoint 目录 {CKPT_DIR} 不存在，无需删除。")


正在删除现有 Checkpoint 目录: /home/aistudio/INTRA/checkpoints/t5gemma2_4b-4b_Lp7...
Checkpoint 目录已删除。请现在再次运行构建 Checkpoint 的单元格 (09e59277)。


### 第3步：玩具演示 (在小数据集上运行模型，验证完整流水线的正确性，避免在大型实验上浪费资源。)

此步骤将运行一个包含 21 篇文档的玩具演示，用于验证整个 INTRA 流水线是否正常工作。这有助于在投入大量计算资源进行完整基准测试之前，快速确认模型是否能成功加载和回答问题。

**硬件要求**：约 16GB 显存。

In [45]:
# ================================================================
# INTRA 复现 - 第3步：玩具演示（强烈建议先运行！）
# ================================================================
# 在 21 篇小文档上验证完整流水线是否正常运行，
# 避免在 80GB 完整基准测试上浪费 GPU 时长。
#
# 硬件要求：约 16GB 显存（无需 CLaRa 1亿文档池）
# 期望结果：模型成功加载，问题被正确回答
# ================================================================

import os, torch

INTRA_DIR = "/home/aistudio/INTRA"

# ---------- 4-A. GPU / 显存合法性检查 ----------
def check_gpu_for_demo():
    if not torch.cuda.is_available():
        raise RuntimeError("未检测到 GPU，玩具演示需要 >= 16GB 显存。")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU VRAM: {vram:.1f} GB")
    if vram < 16:
        raise RuntimeError(f"需要 >= 16GB 显存，当前仅有 {vram:.1f} GB。请切换到 V100-16GB 或 A100 环境。")
    print("显存检查：通过")

check_gpu_for_demo()

# ---------- 4-B. 运行玩具演示脚本 ----------
print("\n正在运行玩具演示（21 篇文档）...")
import subprocess
result = subprocess.run(
    "python scripts/toy_demo.py",
    shell=True, cwd=INTRA_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True
)
print(result.stdout[-4000:])
if result.returncode == 0:
    print("\n玩具演示：通过！")
else:
    print(f"\n玩具演示：失败 (rc={result.returncode})，请查看上方输出。")

# ---------- 4-C. Checkpoint 自检：参数量统计 + 前向传播验证 ----------
print("\n--- 自检：加载 Checkpoint 并执行前向传播 ---")
try:
    import sys
    sys.path.insert(0, INTRA_DIR)
    # 将 INTRA 代码库加入模块搜索路径，并导入模型类
    from intra.model.model import IntraModel
    import json

    CKPT_DIR = os.path.join(INTRA_DIR, "checkpoints", "t5gemma2_4b-4b_Lp7")
    if os.path.isdir(CKPT_DIR):
        # 以 bfloat16 精度加载模型（节省显存）
        device = torch.device("cuda")
        model = IntraModel.from_checkpoint(CKPT_DIR).to(device=device, dtype=torch.bfloat16)
        model.eval()

        # 统计总参数量（4b-4b 架构预期约 9B）
        total_params = sum(p.numel() for p in model.parameters())
        print(f"总参数量: {total_params/1e9:.2f}B（4b-4b 架构预期约 9B）")

        # 用虚拟输入做一次前向传播，验证无 NaN
        with torch.no_grad():
            dummy_ids = torch.ones(1, 16, dtype=torch.long, device=device)
            out = model(input_ids=dummy_ids, decoder_input_ids=dummy_ids[:, :1])
            logits = out.logits if hasattr(out, 'logits') else out[0]
            has_nan = torch.isnan(logits).any().item()
            print(f"前向传播 logits 形状: {logits.shape}")
            print(f"logits 是否含 NaN: {has_nan}（预期为 False）")
            print("自检：通过" if not has_nan else "自检：失败（检测到 NaN！）")
    else:
        print(f"Checkpoint 未找到：{CKPT_DIR}，请先运行第3步。")
except Exception as e:
    print(f"自检出错（可能是路径问题）: {e}")
    print("若玩具演示通过，模型运行正常。")

# ---------- 4-D. 加载模型后的显存占用 ----------
if torch.cuda.is_available():
    mem_alloc = torch.cuda.memory_allocated(0) / 1024**3
    mem_total  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"\n模型加载后显存占用: {mem_alloc:.1f} / {mem_total:.1f} GB")
    print("（模型权重约 18GB；完整 80GB = 模型权重 + 检索向量库）")

print("\n第4步完成。可继续运行第5-7步进行完整基准测试（需要 80GB）。")


RuntimeError: 未检测到 GPU，玩具演示需要 >= 16GB 显存。

### 第4步：完整基准测试 (需要 80GB 显存)

这是 INTRA 论文复现的核心部分，包括重建 CLaRa 1亿文档池、编码向量检索库，并在多个基准测试数据集上评估模型性能。

**硬件要求**：需要至少 70GB (建议 80GB) 显存，通常需要 A100 或 H100 GPU。编码向量库可能需要数小时。

In [ ]:
# ================================================================
# INTRA 复现 - 第5-7步：完整基准测试（需要 80GB 显存）
# ================================================================
# 第5步：重建 CLaRa 1亿条目证据池（结果一致性至关重要！）
# 第6步：编码向量检索库（耗时：A100 上需要数小时）
# 第7步：运行4个基准测试 + 与论文结果对比（需完整 80GB）
# ================================================================
# 硬件检查 - 请先运行此检查：
# ================================================================

import torch, os, subprocess, numpy as np

INTRA_DIR = "/home/aistudio/INTRA"

def check_full_bench_hw():
    if not torch.cuda.is_available():
        raise RuntimeError("未检测到 GPU，完整基准测试需要 80GB 显存（A100/H100）。")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU VRAM: {vram:.1f} GB")
    if vram < 70:
        print(f"警告：完整基准测试需要约 80GB 显存，当前仅有 {vram:.1f}GB。")
        print("可以继续运行第5-6步构建向量库，但评测阶段可能 OOM（显存不足）。")
        print("解决方案：多卡并行、int8 量化，或租用 A100 实例。")
    else:
        print("显存检查：通过，可运行完整基准测试。")

check_full_bench_hw()

# ================================================================
# 第5步：重建 CLaRa 1亿文档池
# ================================================================
# 读取 pool_indices.json（133字节的清单文件）并重建
# apple/CLaRa_multi_stage（HuggingFace）上的语料库。
# 输出：context_pool.json（数百 MB 到几 GB）
#
# 一致性注意事项（避免数据集版本漂移导致结果不可复现）：
#   - 锁定数据集的 revision/commit hash
#   - 锁定 transformers 和 tokenizer 版本（pip freeze）
#   - 不要修改 pool_indices.json 文件
# ================================================================

POOL_FILE = os.path.join(INTRA_DIR, "context_pool.json")

if os.path.exists(POOL_FILE):
    size_mb = os.path.getsize(POOL_FILE) / 1024**2
    print(f"\n[跳过第5步] 文档池已存在，大小: {size_mb:.0f} MB")
else:
    print("\n第5步：重建 CLaRa 1亿文档池...")
    print("  这将从 HuggingFace 下载 apple/CLaRa_multi_stage。")
    print("  目标 token 数：约 1亿（允许去重误差）")
    result = subprocess.run(
        "python -m data.replicate_pool",
        shell=True, cwd=INTRA_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    print(result.stdout[-3000:])
    if result.returncode == 0:
        print("文档池重建完成！")
        # 快速验证文档池文件
        import json
        with open(POOL_FILE) as f:
            pool = json.load(f)
        n_passages = len(pool)
        print(f"  文档池段落数: {n_passages:,}")
        if n_passages < 100_000:
            print("  警告：段落数过少，请检查数据集版本或去重设置。")
        # 抽样检查文档内容
        sample = pool[0]
        print(f"  样本段落（前100字符）: {str(sample)[:100]}")
    else:
        print(f"文档池重建失败 (rc={result.returncode})")

# ================================================================
# 第6步：编码向量检索库（最耗时的步骤）
# ================================================================
# 使用 INTRA checkpoint 的编码器将所有段落编码为
# 归一化的 bf16 向量。输出目录：clara_100M_dedup/
#
# 磁盘：数十 GB | 耗时：A100 上数小时 | 数据类型：bf16
# 显存：约 18GB（模型权重）- 此阶段无需检索向量库
#
# 支持分片编码：多卡时设置 NUM_SHARDS > 1
# ================================================================

BANK_DIR = os.path.join(INTRA_DIR,
    "clara_100M_dedup", "t5gemma2_4b_embeddings_all")

NUM_SHARDS  = 1   # 分片总数，多 GPU 时设置 > 1
SHARD_ID    = 0   # 当前分片索引（从 0 开始）

if os.path.isdir(BANK_DIR):
    files = os.listdir(BANK_DIR)
    print(f"\n[跳过第6步] 向量库目录已存在，包含 {len(files)} 个文件")
else:
    print("\n第6步：编码检索向量库（预计需要数小时）...")
    print(f"  分片总数: {NUM_SHARDS}  |  当前分片: {SHARD_ID}")
    cmd = (f"python scripts/encode_pool.py "
           f"--num_shards {NUM_SHARDS} --shard_id {SHARD_ID}")
    result = subprocess.run(
        cmd, shell=True, cwd=INTRA_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    print(result.stdout[-3000:])
    if result.returncode == 0:
        print("向量编码完成！")
    else:
        print(f"向量编码失败 (rc={result.returncode})")

# 第6步自检：验证向量嵌入的有效性
if os.path.isdir(BANK_DIR):
    emb_files = sorted([f for f in os.listdir(BANK_DIR) if f.endswith('.npy')])
    print(f"\n[向量库自检] 找到 {len(emb_files)} 个嵌入分片文件")
    if emb_files:
        sample_path = os.path.join(BANK_DIR, emb_files[0])
        emb = np.load(sample_path)
        print(f"  形状: {emb.shape}  数据类型: {emb.dtype}")
        norms = np.linalg.norm(emb[:100].astype(np.float32), axis=-1)
        mean_norm = norms.mean()
        has_nan = np.isnan(emb[:100]).any()
        print(f"  前100个向量平均范数: {mean_norm:.4f}（归一化后应约为 1.0）")
        print(f"  含 NaN: {has_nan}（预期为 False）")
        if abs(mean_norm - 1.0) > 0.05:
            print("  警告：范数偏离 1.0，请检查是否应用了归一化！")

# ================================================================
# 第7步：运行基准测试评估
# ================================================================
# 将完整检索库加载进 GPU 显存 -> 需要 80GB 显存
# 在4个数据集上评测，报告 EM 和 F1 分数。
#
# 论文目标结果（EM / F1）：
#   HotpotQA：EM=46.4% / F1=58.0%
#   2WikiMQA：EM=49.2% / F1=53.2%
#   MuSiQue： EM=14.0% / F1=23.0%
#   NQ：     EM=51.2% / F1=60.3%
# ================================================================

print("\n第7步：运行完整基准测试评估...")
print("  这将把完整检索向量库加载进显存（额外约 8-32 GB）。")
print("  总显存需求：约18GB（模型）+ 约8-32GB（向量库）= 最多80GB")

result = subprocess.run(
    "python tests/eval_paper_benchmarks.py",
    shell=True, cwd=INTRA_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
output = result.stdout
print(output[-4000:])

# 解析并比较复现结果与论文结果
print("\n" + "=" * 62)
print("论文结果 vs 复现结果")
print("=" * 62)
print(f"{'数据集':<12} {'EM(论文)':<12} {'F1(论文)':<12}  说明")
print("-" * 62)
paper_scores = [
    ("HotpotQA", 46.4, 58.0, "二跳多步推理"),
    ("2WikiMQA",  49.2, 53.2, "结构化多跳推理"),
    ("MuSiQue",   14.0, 23.0, "最难，2-4跳推理"),
    ("NQ",        51.2, 60.3, "单跳开放域问答"),
]
for bm, em, f1, note in paper_scores:
    print(f"  {bm:<10} {em:<10.1f}% {f1:<10.1f}%  {note}")
print("-" * 62)
print("注意：±1-2分的差异属于正常复现误差范围。")
print("若 MuSiQue > 40%，请检查数据/评测流水线是否有 bug。")
print("=" * 62)
print("\n第5-7步完成。INTRA 完整复现成功！")on done!")


### 导出 INTRA 文件夹到本地

如果您想将当前 `INTRA` 文件夹（包含所有克隆的文件、下载的 LFS 对象、以及构建好的 Checkpoint）保存到本地，可以将其压缩成 `.zip` 文件并下载。请注意，压缩大型文件夹可能需要一些时间。

In [8]:
import os
import shutil
from google.colab import files

INTRA_DIR = "/home/aistudio/INTRA"
ARCHIVE_NAME = "INTRA_archive"
ARCHIVE_PATH = "/content/" + ARCHIVE_NAME

if os.path.exists(INTRA_DIR):
    print(f"正在将 {INTRA_DIR} 压缩到 {ARCHIVE_PATH}.zip...")
    # 创建压缩文件
    shutil.make_archive(ARCHIVE_PATH, 'zip', INTRA_DIR)
    print("压缩完成，准备下载。")
    # 下载文件
    files.download(ARCHIVE_PATH + '.zip')
    print("文件下载请求已发出。")
else:
    print(f"错误：INTRA 文件夹未找到于: {INTRA_DIR}，无法压缩。")


正在将 /home/aistudio/INTRA 压缩到 /content/INTRA_archive.zip...


KeyboardInterrupt: 

### 清理笔记本元数据以彻底解决 GitHub 推送问题 (再次提供)

如果 GitHub 推送失败，并出现 `Invalid Notebook: the 'state' key is missing from 'metadata.widgets'` 错误，请运行此单元格来清理笔记本的元数据。运行后，请**务必手动保存 Colab 笔记本（文件 -> 保存）**，然后再次尝试推送到 GitHub。

In [49]:
import json
import os
import io
from google.colab import _message as _colab_message

def get_notebook_path():
    try:
        import re
        cmdline = open('/proc/self/cmdline', 'rb').read().decode('utf-8')
        match = re.search(r'--notebook-path=([^\s]+)', cmdline)
        if match:
            return match.group(1)
    except Exception as e:
        print(f"无法自动获取笔记本路径: {e}")
    return None

notebook_path = get_notebook_path()

if not notebook_path or not notebook_path.endswith('.ipynb'):
    print("自动路径检测失败或不完整，尝试通用路径...")
    possible_paths = []
    for root, _, files in os.walk('/content'):
        for f in files:
            if f.endswith('.ipynb'):
                possible_paths.append(os.path.join(root, f))

    if possible_paths:
        print("可能找到的笔记本文件：")
        for p in possible_paths:
            print(f"  - {p}")
        if len(possible_paths) == 1:
            notebook_path = possible_paths[0]
            print(f"自动选择：{notebook_path}")
        else:
            print("请从上方列表中选择正确的笔记本路径并手动赋值给 `notebook_path` 变量。")
    else:
        print("未在 /content 及其子目录中找到 .ipynb 文件。请手动指定笔记本路径。")
        notebook_path = "/content/Your_Notebook_Name.ipynb" # Placeholder


if notebook_path and os.path.exists(notebook_path):
    try:
        with open(notebook_path, 'r', encoding='utf-8') as f:
            notebook_content = json.load(f)

        if 'metadata' in notebook_content and 'widgets' in notebook_content['metadata']:
            print("检测到 'metadata.widgets'。正在尝试清理...")
            del notebook_content['metadata']['widgets']
            print("已移除 'metadata.widgets'。")

            with open(notebook_path, 'w', encoding='utf-8') as f:
                json.dump(notebook_content, f, indent=4)
            print(f"清理后的笔记本已保存到 {notebook_path}")
            print("\n重要：请手动保存 Colab 笔记本（文件 -> 保存），然后再次尝试推送到 GitHub。")
        else:
            print("笔记本中未检测到 'metadata.widgets' 或其格式正常，无需清理。")
            print("如果 GitHub 仍报错，请检查其他可能原因。")
    except json.JSONDecodeError:
        print(f"错误：无法解析笔记本文件 {notebook_path} 为 JSON。文件可能已损坏。")
    except Exception as e:
        print(f"清理笔记本元数据时发生错误: {e}")
else:
    print(f"错误：指定的笔记本路径 {notebook_path} 不存在或无效。请手动指定正确的路径。")

自动路径检测失败或不完整，尝试通用路径...
可能找到的笔记本文件：
  - /content/drive/MyDrive/Colab Notebooks/INTRA-repro.ipynb
自动选择：/content/drive/MyDrive/Colab Notebooks/INTRA-repro.ipynb
笔记本中未检测到 'metadata.widgets' 或其格式正常，无需清理。
如果 GitHub 仍报错，请检查其他可能原因。
